# Deep-SSFS — Notebook 2: Deep Selector Architecture, Training & Hyperparameter Tuning

contains the full Deep-SSFS training

## 1. Environment Setup

In [ ]:
!pip install -q numpy==1.26.4 pandas==2.1.4 scikit-learn-extra==0.3.0

In [ ]:
import os
import json
import random
import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_olivetti_faces
from sklearn_extra.cluster import KMedoids
from scipy.linalg import eigh

from tensorflow.keras.datasets import mnist, fashion_mnist, cifar10

print("All imports successful.")

## 2. Global Configuration



In [ ]:
# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

SEED       = 42
RESULTS_DIR = './results/'
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASET_PARAMS = {
    'Olivetti Faces': {'num_features': 30, 'num_clusters': 40, 'k': 6},
    'MNIST':          {'num_features': 50, 'num_clusters': 10, 'k': 5},
    'Fashion-MNIST':  {'num_features': 50, 'num_clusters': 10, 'k': 5},
    'CIFAR-10':       {'num_features': 30, 'num_clusters': 10, 'k': 6},
}

# Known Classic-SSFS 1-NN benchmark scores for comparison.
OLD_BENCHMARK_SCORES = {
    'MNIST_2000':           0.6750,
    'MNIST_6000':           0.7450,
    'MNIST_10000':          0.6723,
    'Fashion-MNIST_2000':   0.6975,
    'Fashion-MNIST_6000':   0.7375,
    'Fashion-MNIST_10000':  0.7038,
    'CIFAR-10_1000':        None,
    'Olivetti_Faces_400':   None,
}

# Dataset sample counts per experimental session.
DATASET_N_SAMPLES = {
    'Olivetti Faces': None,
    'MNIST':          2000,
    'Fashion-MNIST':  2000,
    'CIFAR-10':       2000,
}

# Stability/training knobs (final values from Session 5 / fashion-grid.ipynb).
N_RESAMPLES      = 10    # inner SSFS resample passes per stability run
N_STABILITY_RUNS = 5     # outer stability aggregation runs
NUM_FULL_RUNS    = 3     # independent evaluation cycles for mean/std

# Base model hyperparameters (overridden by grid-search winner).
MODEL_ARGS = {
    'epochs':        80,     # increased from 50 for deeper convergence
    'batch_size':    128,
    'lr':            0.001,
    'lasso_lambda':  0.1,
    'use_scheduler': True,
    'weight_decay':  1e-4,   # AdamW L2 regularisation
    'patience':      10,     # early-stopping patience (epochs without improvement)
    'dropout':       0.3,
}

# Session-level dataset lists.
SESSION_1_DATASETS = ['Olivetti Faces', 'MNIST', 'Fashion-MNIST', 'CIFAR-10']
SESSION_2_DATASETS = ['Olivetti Faces', 'MNIST', 'Fashion-MNIST']
SESSION_3_DATASETS = ['MNIST', 'Fashion-MNIST']
SESSION_4_DATASETS = ['MNIST', 'Fashion-MNIST']
SESSION_5_DATASETS = ['MNIST', 'Fashion-MNIST']

SESSION_3_N_SAMPLES = 6000
SESSION_4_N_SAMPLES = 6000
SESSION_5_SAMPLE_SIZES = [2000, 4000, 6000, 10000]   # scalability sweep


# ---- Reproducibility ----
def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ARGS['device'] = DEVICE

def _ts():
    return datetime.datetime.now().strftime('%H:%M:%S')

print(f"[{_ts()}] Device      : {DEVICE}")
print(f"[{_ts()}] GPU count   : {torch.cuda.device_count()}")
print(f"[{_ts()}] Results dir : {RESULTS_DIR}")

## 3. Data Loaders & Preprocessing



In [ ]:
def _subsample(X, y, n_samples):
    if n_samples is not None and n_samples < X.shape[0]:
        idx = np.random.choice(X.shape[0], n_samples, replace=False)
        return X[idx], y[idx]
    return X, y


def load_olivetti_faces(n_samples=None):
    print(f"[{_ts()}] Loading Olivetti Faces...")
    data = fetch_olivetti_faces(shuffle=True, random_state=SEED)
    return _subsample(data.data, data.target, n_samples)


def load_mnist_data(n_samples=2000):
    print(f"[{_ts()}] Loading MNIST (n={n_samples})...")
    (x_tr, y_tr), (x_te, y_te) = mnist.load_data()
    X = np.concatenate([x_tr, x_te]).reshape(-1, 784).astype('float32') / 255.0
    y = np.concatenate([y_tr, y_te])
    return _subsample(X, y, n_samples)


def load_fashion_mnist_data(n_samples=2000):
    print(f"[{_ts()}] Loading Fashion-MNIST (n={n_samples})...")
    (x_tr, y_tr), (x_te, y_te) = fashion_mnist.load_data()
    X = np.concatenate([x_tr, x_te]).reshape(-1, 784).astype('float32') / 255.0
    y = np.concatenate([y_tr, y_te])
    return _subsample(X, y, n_samples)


def load_cifar10_data(n_samples=2000):
    print(f"[{_ts()}] Loading CIFAR-10 (n={n_samples})...")
    (x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
    X = np.concatenate([x_tr, x_te]).reshape(-1, 3072).astype('float32') / 255.0
    y = np.concatenate([y_tr, y_te]).flatten()
    return _subsample(X, y, n_samples)


LOADER_MAP = {
    'Olivetti Faces': load_olivetti_faces,
    'MNIST':          load_mnist_data,
    'Fashion-MNIST':  load_fashion_mnist_data,
    'CIFAR-10':       load_cifar10_data,
}


def load_dataset(name, n_samples=None):
    loader = LOADER_MAP[name]
    return loader(n_samples) if n_samples is not None else loader()


def prepare_data(X, y, test_size=0.2, seed=SEED):
    """
    Stratified 80/20 split; StandardScaler fitted only on train.
    Returns scaled matrices, labels, and unscaled raw arrays.
    """
    X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr_raw)
    X_te   = scaler.transform(X_te_raw)
    return X_tr, X_te, y_tr, y_te, X_tr_raw, X_te_raw


print("Data loaders defined.")

## 4. Evaluation & Persistence Utilities

In [ ]:
def evaluation(X_train, y_train, X_test, y_test):
    """
    1-Nearest-Neighbour evaluation.
    Returns dict with 'train' and 'test' accuracy.
    """
    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(X_train, y_train)
    return {
        'train': float(accuracy_score(y_train, knn.predict(X_train))),
        'test':  float(accuracy_score(y_test,  knn.predict(X_test))),
    }


def save_results(df, filename, extra_json=None, json_filename=None,
                 save_dir=RESULTS_DIR):
    """
    Save a results DataFrame to a timestamped CSV and an optional JSON sidecar.
    """
    ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    stem = filename.rsplit('.', 1)[0]

    csv_path = os.path.join(save_dir, f"{stem}_{ts}.csv")
    try:
        df.to_csv(csv_path, index=False)
        print(f"   Saved CSV  -> {csv_path}")
    except Exception as e:
        print(f"   CSV save failed: {e}")

    if extra_json is not None and json_filename is not None:
        json_stem = json_filename.rsplit('.', 1)[0]
        json_path = os.path.join(save_dir, f"{json_stem}_{ts}.json")
        try:
            with open(json_path, 'w') as fh:
                json.dump(extra_json, fh, indent=4)
            print(f"   Saved JSON -> {json_path}")
        except Exception as e:
            print(f"   JSON save failed: {e}")


print("Evaluation and persistence utilities defined.")

## 5. Deep Selector Architecture



In [ ]:
class FeatureSelectionMLP(nn.Module):
    """
    MLP with a learnable input gate for feature selection.

    A parameter vector `selection_weights` (shape: input_dim) multiplies
    each input feature before the first hidden layer. L1 regularisation
    on these weights enforces sparsity, effectively selecting informative
    features while zeroing out irrelevant ones.

    """

    def __init__(self, input_dim: int, n_clusters: int,
                 hidden_dims=None, dropout: float = 0.3):
        super().__init__()
        assert isinstance(input_dim, (int, np.integer)), (
            f"input_dim must be int, got {type(input_dim)}. Pass X.shape[1]."
        )
        if hidden_dims is None:
            hidden_dims = [512, 256, 128]

        self.selection_weights = nn.Parameter(torch.ones(input_dim))

        layers, curr = [], input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(curr, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(p=dropout),
            ]
            curr = h
        layers.append(nn.Linear(curr, n_clusters))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x * torch.abs(self.selection_weights))

    def get_regularization_loss(self):
        """L1 (Lasso) penalty on the input gate weights."""
        return torch.sum(torch.abs(self.selection_weights))

    def get_feature_importances(self):
        """Return absolute gate weights as a numpy array."""
        return torch.abs(self.selection_weights).detach().cpu().numpy()




In [ ]:
class DeepSelectorWrapper:
    """
    Training harness for FeatureSelectionMLP.

    Implements AdamW optimisation, CosineAnnealing LR scheduling,
    gradient clipping, and patience-based early stopping.
    """

    def __init__(self, n_features, n_clusters,
                 epochs=80, batch_size=128,
                 lr=0.001, lasso_lambda=0.01,
                 device='cpu', use_scheduler=True,
                 weight_decay=1e-4, patience=10,
                 dropout=0.3, verbose=False):
        self.n_features    = n_features
        self.n_clusters    = n_clusters
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.lr            = lr
        self.lasso_lambda  = lasso_lambda
        self.device        = device
        self.use_scheduler = use_scheduler
        self.weight_decay  = weight_decay
        self.patience      = patience
        self.dropout       = dropout
        self.verbose       = verbose
        self.feature_importances_ = None
        self.model = None

    def fit(self, X, y):
        n_feat = X.shape[1]

        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_t = torch.tensor(y, dtype=torch.long).to(self.device)
        loader = DataLoader(
            TensorDataset(X_t, y_t),
            batch_size=self.batch_size,
            shuffle=True,
            drop_last=True,
        )

        model = FeatureSelectionMLP(
            input_dim=n_feat,
            n_clusters=self.n_clusters,
            dropout=self.dropout,
        ).to(self.device)

        optimizer = optim.AdamW(
            model.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        criterion = nn.CrossEntropyLoss()
        scheduler = (
            optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=self.epochs, eta_min=1e-6
            ) if self.use_scheduler else None
        )

        best_loss, best_weights, patience_ctr = float('inf'), None, 0

        model.train()
        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for bX, by in loader:
                optimizer.zero_grad()
                loss = (
                    criterion(model(bX), by)
                    + self.lasso_lambda * model.get_regularization_loss()
                )
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                epoch_loss += loss.item()

            if scheduler:
                scheduler.step()

            if epoch_loss < best_loss - 1e-6:
                best_loss    = epoch_loss
                best_weights = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= self.patience:
                    if self.verbose:
                        print(f"    Early stop at epoch {epoch+1}.")
                    break

        if best_weights is not None:
            model.load_state_dict(best_weights)

        self.model                = model
        self.feature_importances_ = model.get_feature_importances()
        return self

    def save_weights(self, path: str):
        if self.model is None:
            raise RuntimeError("Call fit() before save_weights().")
        torch.save(self.model.state_dict(), path)




## 7. SSFS Core Algorithm



In [ ]:
class SSFS(BaseEstimator, TransformerMixin):
    """
    Self-supervised Spectral Feature Selection with a Deep surrogate model.


    """

    def __init__(self, n_features=100, n_clusters=20,
                 model_name='deep_selector', sigma=1.0,
                 k=5, n_resamples=10, args_model=None, verbose=False):
        self.n_features  = n_features
        self.n_clusters  = n_clusters
        self.model_name  = model_name
        self.sigma       = sigma
        self.k           = k
        self.n_resamples = n_resamples
        self.args_model  = args_model if args_model is not None else {}
        self.verbose     = verbose
        self.selected_features_ = None
        self.final_scores_      = None

    def _get_model_instance(self):
        if self.model_name == 'deep_selector':
            return DeepSelectorWrapper(
                n_features=self.n_features,
                n_clusters=self.n_clusters,
                verbose=self.verbose,
                **self.args_model,
            )
        if self.model_name == 'ridge':
            from sklearn.linear_model import Ridge
            return Ridge(**self.args_model)
        raise ValueError(f"Unknown model: '{self.model_name}'")

    def _compute_W(self, X):
        """Adaptive-bandwidth Gaussian k-NN affinity matrix."""
        n      = X.shape[0]   # rows = samples
        nn_fit = NearestNeighbors(n_neighbors=self.k + 1, algorithm='auto').fit(X)
        distances, indices = nn_fit.kneighbors(X)
        sigmas = np.maximum(distances[:, -1], 1e-8)
        W      = np.zeros((n, n), dtype=np.float32)
        for i in range(n):
            for j in indices[i, 1:]:
                d_sq     = np.sum((X[i] - X[j]) ** 2)
                w        = np.exp(-d_sq / (sigmas[i] * sigmas[j]))
                W[i, j]  = W[j, i] = w
        return W

    def _get_feature_importances(self, model):
        if isinstance(model, DeepSelectorWrapper):
            return model.feature_importances_
        if hasattr(model, 'coef_'):
            c = model.coef_
            return np.abs(c) if c.ndim == 1 else np.linalg.norm(c, axis=0)
        return np.zeros(self.n_features)

    def fit(self, X, y=None):
        # X.shape[0] is a plain int that cannot be unpacked into two variables.
        n_samples, n_orig = X.shape

        feature_scores = np.zeros(n_orig, dtype=np.float64)

        W          = self._compute_W(X)
        D          = W.sum(axis=1)
        D_inv_sqrt = np.where(D > 0, D ** -0.5, 0.0)
        D_mat      = np.diag(D_inv_sqrt)
        L          = np.eye(n_samples, dtype=np.float32) - D_mat @ W @ D_mat

        n_eigs = min(self.n_clusters * 2, n_samples - 1)
        _, eigvecs = eigh(L, subset_by_index=[1, n_eigs])
        norms      = np.linalg.norm(eigvecs, axis=1, keepdims=True)
        eigvecs    = eigvecs / np.maximum(norms, 1e-8)

        for i in range(self.n_resamples):
            sub_size = int(0.8 * n_samples)
            idx      = np.random.choice(n_samples, sub_size, replace=False)
            X_s, ev_s = X[idx], eigvecs[idx]

            y_pseudo = KMedoids(
                n_clusters=self.n_clusters, random_state=i, method='alternate'
            ).fit(ev_s).labels_

            model = self._get_model_instance()
            model.fit(X_s, y_pseudo)
            feature_scores += self._get_feature_importances(model)

        self.final_scores_      = feature_scores / self.n_resamples
        self.selected_features_ = np.argsort(self.final_scores_)[::-1][:self.n_features]
        return self

    def transform(self, X):
        if self.selected_features_ is None:
            raise RuntimeError("Call fit() before transform().")
        return X[:, self.selected_features_]




## 8. Experiment Helper Functions



In [ ]:
def simple_grid_search(X_train, y_train, dataset_params, test_size=0.2):
    """
    Minimal grid search for Deep-SSFS: sweeps lr × lasso_lambda using
    a single-resample SSFS pass and 1-NN validation accuracy.

    Returns the best (lr, lasso_lambda) dict.
    """
    print(f"[{_ts()}] Starting grid search...")

    X_sub_tr, X_val, y_sub_tr, y_val = train_test_split(
        X_train, y_train, test_size=test_size, stratify=y_train, random_state=SEED
    )

    grid = [
        {'lr': 0.001,  'lasso_lambda': 0.1},
        {'lr': 0.001,  'lasso_lambda': 0.01},
        {'lr': 0.0005, 'lasso_lambda': 0.1},
        {'lr': 0.0005, 'lasso_lambda': 0.01},
        {'lr': 0.001,  'lasso_lambda': 0.005},
        {'lr': 0.001,  'lasso_lambda': 0.05},
    ]

    best_acc, best_hp = 0.0, grid[0]

    for hp in grid:
        print(f"  Testing lr={hp['lr']}, lasso={hp['lasso_lambda']}...", end='')
        model_args = {**MODEL_ARGS, 'lr': hp['lr'], 'lasso_lambda': hp['lasso_lambda'],
                      'device': DEVICE}
        m = SSFS(
            n_features=dataset_params['num_features'],
            n_clusters=dataset_params['num_clusters'],
            model_name='deep_selector',
            k=dataset_params.get('k', 5),
            args_model=model_args,
            n_resamples=1,
        )
        m.fit(X_sub_tr)

        if m.selected_features_ is not None and len(m.selected_features_) > 0:
            scores = evaluation(
                X_sub_tr[:, m.selected_features_], y_sub_tr,
                X_val[:,   m.selected_features_], y_val,
            )
            acc = scores['test']
        else:
            acc = 0.0

        print(f" val_1NN={acc:.4f}")
        if acc > best_acc:
            best_acc, best_hp = acc, hp

    print(f"[{_ts()}] Best HP: {best_hp}  (val 1-NN: {best_acc:.4f})")
    return best_hp


def run_stability_selection(X_train, dataset_params, hyperparams,
                             n_stability_runs, n_resamples,
                             extra_model_args=None):
    """
    Outer stability selection: runs `n_stability_runs` independent SSFS
    passes and averages feature importance scores.

    """
    model_args = {
        **MODEL_ARGS,
        **(extra_model_args or {}),
        'lr':           hyperparams['lr'],
        'lasso_lambda': hyperparams['lasso_lambda'],
        'device':       DEVICE,
    }

    n_features = X_train.shape[1]
    cumulative = np.zeros(n_features, dtype=np.float64)
    valid_runs = 0
    last_ssfs  = None

    for i in range(n_stability_runs):
        print(f"    Stability run {i+1:>2}/{n_stability_runs} ...", end='\r')
        try:
            m = SSFS(
                n_features=dataset_params['num_features'],
                n_clusters=dataset_params['num_clusters'],
                model_name='deep_selector',
                k=dataset_params.get('k', 5),
                args_model=model_args,
                n_resamples=n_resamples,
            )
            m.fit(X_train)
            if m.final_scores_ is not None:
                cumulative += m.final_scores_
                valid_runs += 1
                last_ssfs   = m
        except Exception as exc:
            print(f"\n    [WARNING] Stability run {i+1} failed: {exc}")

    print()

    if valid_runs == 0:
        raise RuntimeError("All stability runs failed — check configuration.")

    avg_scores = cumulative / valid_runs
    selected   = np.argsort(avg_scores)[::-1][:dataset_params['num_features']]
    return avg_scores, selected, last_ssfs




## 9. Session 1 — Naive Baseline



In [ ]:
NAIVE_MODEL_ARGS = {
    'lr':            0.001,
    'lasso_lambda':  0.1,
    'epochs':        30,
    'batch_size':    64,
    'device':        DEVICE,
    'use_scheduler': False,
    'weight_decay':  1e-4,
    'patience':      10,
    'dropout':       0.3,
}

print("SESSION 1: NAIVE BASELINE")
rows_s1 = []

for name in SESSION_1_DATASETS:
    try:
        n_samples = DATASET_N_SAMPLES.get(name)
        params    = DATASET_PARAMS[name]
        print(f"\n[{name}]")

        X, y = load_dataset(name, n_samples)
        X_train, X_test, y_train, y_test, _, _ = prepare_data(X, y)

        model = SSFS(
            n_features=params['num_features'],
            n_clusters=params['num_clusters'],
            model_name='deep_selector',
            k=params.get('k', 5),
            args_model=NAIVE_MODEL_ARGS,
            n_resamples=1,
        )
        model.fit(X_train)

        sel    = model.selected_features_
        scores = evaluation(X_train[:, sel], y_train, X_test[:, sel], y_test)
        print(f"  1-NN Train: {scores['train']:.4f}  |  Test: {scores['test']:.4f}")

        rows_s1.append({
            'Dataset':  name,
            'Session':  '1 (Naive)',
            'Train Acc': round(scores['train'], 4),
            'Test Acc':  round(scores['test'],  4),
        })

    except Exception:
        import traceback
        traceback.print_exc()

df_s1 = pd.DataFrame(rows_s1)
print("\nSESSION 1 RESULTS")
print(df_s1.to_string(index=False))
save_results(df_s1, 'session1_naive_results.csv')

## 10. Session 2 — Tuned Hyperparameters + Stability Selection



In [ ]:
print("SESSION 2: TUNED + STABILITY SELECTION")
rows_s2          = []
best_hyperparams = {}

for name in SESSION_2_DATASETS:
    try:
        n_samples = DATASET_N_SAMPLES.get(name)
        params    = DATASET_PARAMS[name]
        print(f"\n[{name}]")

        X, y = load_dataset(name, n_samples)
        X_train, X_test, y_train, y_test, _, _ = prepare_data(X, y)

        hp = simple_grid_search(X_train, y_train, params)
        best_hyperparams[name] = hp

        avg_scores, selected, _ = run_stability_selection(
            X_train, params, hp,
            n_stability_runs=N_STABILITY_RUNS,
            n_resamples=N_RESAMPLES,
        )

        scores = evaluation(
            X_train[:, selected], y_train,
            X_test[:,  selected], y_test,
        )

        bench_key = f"{name}_{n_samples}"
        old_score = OLD_BENCHMARK_SCORES.get(bench_key)
        diff      = round(scores['test'] - old_score, 4) if old_score is not None else None

        print(f"  1-NN Test: {scores['test']:.4f}  |  Benchmark: {old_score}  |  Diff: {diff}")

        rows_s2.append({
            'Dataset':       name,
            'Num Features':  params['num_features'],
            'Best Params':   str(hp),
            'Test (1-NN)':   round(scores['test'], 4),
            'Old Benchmark': old_score,
            'Diff':          diff,
        })

    except Exception:
        import traceback
        traceback.print_exc()

df_s2 = pd.DataFrame(rows_s2)
print("\nSESSION 2 RESULTS")
print(df_s2[['Dataset', 'Test (1-NN)', 'Old Benchmark', 'Diff']].to_string(index=False))
save_results(
    df_s2,
    'session2_tuned_results.csv',
    extra_json=best_hyperparams,
    json_filename='best_hyperparams_session2.json',
)

## 11. Session 3 — Scalability Test (6 K samples)



In [ ]:
# Load saved hyperparams from Session 2 if not already in memory.
_hp_path = os.path.join(RESULTS_DIR, 'best_hyperparams_session2.json')
if 'best_hyperparams' not in dir() or not best_hyperparams:
    try:
        with open(_hp_path) as fh:
            best_hyperparams = json.load(fh)
        print(f"Loaded hyperparams from {_hp_path}")
    except Exception:
        best_hyperparams = {}
        print("No saved hyperparams — using MODEL_ARGS defaults.")

print(f"SESSION 3: SCALABILITY TEST (n={SESSION_3_N_SAMPLES})")
rows_s3 = []

for name in SESSION_3_DATASETS:
    try:
        params = DATASET_PARAMS[name]
        print(f"\n[{name}]")

        X, y = load_dataset(name, SESSION_3_N_SAMPLES)
        X_train, X_test, y_train, y_test, _, _ = prepare_data(X, y)

        hp = best_hyperparams.get(
            name, {'lr': MODEL_ARGS['lr'], 'lasso_lambda': MODEL_ARGS['lasso_lambda']}
        )
        print(f"  Hyperparams: {hp}")

        avg_scores, selected, _ = run_stability_selection(
            X_train, params, hp,
            n_stability_runs=N_STABILITY_RUNS,
            n_resamples=N_RESAMPLES,
        )

        scores = evaluation(
            X_train[:, selected], y_train,
            X_test[:,  selected], y_test,
        )

        bench_key = f"{name}_{SESSION_3_N_SAMPLES}"
        old_score = OLD_BENCHMARK_SCORES.get(bench_key)
        diff      = round(scores['test'] - old_score, 4) if old_score is not None else None

        print(f"  1-NN Test: {scores['test']:.4f}  |  Benchmark: {old_score}  |  Diff: {diff}")
        rows_s3.append({
            'Dataset':       name,
            'Samples':       SESSION_3_N_SAMPLES,
            'Num Features':  params['num_features'],
            'Test (1-NN)':   round(scores['test'], 4),
            'Old Benchmark': old_score,
            'Diff':          diff,
        })

    except Exception:
        import traceback
        traceback.print_exc()

df_s3 = pd.DataFrame(rows_s3)
print("\nSESSION 3 RESULTS")
print(df_s3.to_string(index=False))
save_results(df_s3, f'session3_scalability_{SESSION_3_N_SAMPLES}samples.csv')

> **Observation:** Accuracy can drop when scaling from 2K to 6K samples because the hyperparameters were tuned for the smaller regime. The Lasso penalty that worked well at 2K may be too restrictive (or too permissive) for a 6K batch. Session 4 re-tunes specifically at this scale.

## 12. Session 4 — Re-tune at Scale (6 K)

 fresh grid search at `SESSION_4_N_SAMPLES=6000` to find hyperparameters optimised for the larger data regime.

In [ ]:
print(f"SESSION 4: RE-TUNE AT SCALE (n={SESSION_4_N_SAMPLES})")
rows_s4           = []
best_hyperparams_s4 = {}

for name in SESSION_4_DATASETS:
    try:
        params = DATASET_PARAMS[name]
        print(f"\n[{name}]")

        X, y = load_dataset(name, SESSION_4_N_SAMPLES)
        X_train, X_test, y_train, y_test, _, _ = prepare_data(X, y)

        hp = simple_grid_search(X_train, y_train, params)
        best_hyperparams_s4[name] = hp

        avg_scores, selected, _ = run_stability_selection(
            X_train, params, hp,
            n_stability_runs=N_STABILITY_RUNS,
            n_resamples=N_RESAMPLES,
        )

        scores = evaluation(
            X_train[:, selected], y_train,
            X_test[:,  selected], y_test,
        )

        bench_key = f"{name}_{SESSION_4_N_SAMPLES}"
        old_score = OLD_BENCHMARK_SCORES.get(bench_key)
        diff      = round(scores['test'] - old_score, 4) if old_score is not None else None

        print(f"  1-NN Test: {scores['test']:.4f}  |  Benchmark: {old_score}  |  Diff: {diff}")
        rows_s4.append({
            'Dataset':       name,
            'Samples':       SESSION_4_N_SAMPLES,
            'Best Params':   str(hp),
            'Test (1-NN)':   round(scores['test'], 4),
            'Old Benchmark': old_score,
            'Diff':          diff,
        })

    except Exception:
        import traceback
        traceback.print_exc()

df_s4 = pd.DataFrame(rows_s4)
print("\nSESSION 4 RESULTS")
print(df_s4.to_string(index=False))
save_results(
    df_s4,
    f'session4_retune_{SESSION_4_N_SAMPLES}samples.csv',
    extra_json=best_hyperparams_s4,
    json_filename=f'best_hyperparams_session4_{SESSION_4_N_SAMPLES}.json',
)

## 13. Session 5 — Large-Scale Final Run (Scalability Sweep)

The primary experimental contribution of the paper. For each `(dataset, n_samples)` combination in `SESSION_5_SAMPLE_SIZES`:
1. Per-run grid search.
2. `NUM_FULL_RUNS` independent stability-selection + evaluation cycles.
3. Mean ± std of 1-NN test accuracy reported.
4. Model weights, JSON feature indices, and CSV saved per run.

This covers the full scalability curve from 2K → 10K samples for MNIST and Fashion-MNIST.

In [ ]:
print(f"[{_ts()}] SESSION 5: LARGE-SCALE FINAL RUN")

rows_s5             = []
best_hyperparams_s5 = {}

for name in SESSION_5_DATASETS:
    for n_samples in SESSION_5_SAMPLE_SIZES:
        try:
            print(f"\n{'='*60}")
            print(f"[{_ts()}] Dataset: {name} | Samples: {n_samples}")
            print(f"{'='*60}")

            params = DATASET_PARAMS[name]

            X, y = load_dataset(name, n_samples)
            X_train, X_test, y_train, y_test, _, _ = prepare_data(X, y)
            print(f"[{_ts()}] Train: {X_train.shape}  Test: {X_test.shape}")

            hp_optimal = simple_grid_search(X_train, y_train, params)

            acc_train_list, acc_test_list = [], []
            best_selected, best_acc, best_ssfs = None, 0.0, None

            print(f"[{_ts()}] Running {NUM_FULL_RUNS} evaluation cycles "
                  f"(lr={hp_optimal['lr']}, lasso={hp_optimal['lasso_lambda']})")

            for run_idx in range(NUM_FULL_RUNS):
                print(f"  [Cycle {run_idx+1}/{NUM_FULL_RUNS}]")

                avg_scores, selected, last_ssfs = run_stability_selection(
                    X_train, params, hp_optimal,
                    n_stability_runs=N_STABILITY_RUNS,
                    n_resamples=N_RESAMPLES,
                    extra_model_args={'batch_size': MODEL_ARGS['batch_size']},
                )

                scores = evaluation(
                    X_train[:, selected], y_train,
                    X_test[:,  selected], y_test,
                )
                acc_train_list.append(scores['train'])
                acc_test_list.append(scores['test'])
                print(f"    Train: {scores['train']:.4f}  |  Test (1-NN): {scores['test']:.4f}")

                if scores['test'] > best_acc:
                    best_acc, best_selected, best_ssfs = scores['test'], selected, last_ssfs

            mean_train = float(np.mean(acc_train_list))
            mean_test  = float(np.mean(acc_test_list))
            std_test   = float(np.std(acc_test_list))

            bench_key = f"{name}_{n_samples}"
            old_score = OLD_BENCHMARK_SCORES.get(bench_key)
            diff      = round(mean_test - old_score, 4) if old_score is not None else None

            print(f"[{_ts()}] Mean Train: {mean_train:.4f}  |  "
                  f"Mean Test: {mean_test:.4f} ± {std_test:.4f}  "
                  f"Benchmark: {old_score}  Diff: {diff}")

            ts_now    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            safe_name = name.replace(' ', '_').replace('-', '_')
            base      = f"DeepSSFS_{safe_name}_{n_samples}samples"

            pt_path = os.path.join(RESULTS_DIR, f"{base}_{ts_now}.pt")
            try:
                weight_dict = {
                    'selection_weights': torch.tensor(
                        best_ssfs.final_scores_ if best_ssfs is not None else []
                    )
                }
                torch.save(weight_dict, pt_path)
                print(f"   Saved weights -> {pt_path}")
            except Exception as e:
                print(f"   Weight save failed: {e}")

            selected_list = best_selected.tolist() if best_selected is not None else []
            json_payload  = {
                bench_key: {
                    'dataset':           name,
                    'n_samples':         n_samples,
                    'lr':                hp_optimal['lr'],
                    'lasso_lambda':      hp_optimal['lasso_lambda'],
                    'selected_features': selected_list,
                }
            }
            best_hyperparams_s5[bench_key] = json_payload[bench_key]

            row = {
                'Dataset':          name,
                'Samples':          n_samples,
                'Num Features':     params['num_features'],
                'Mean Train':       round(mean_train, 4),
                'Mean Test (1-NN)': round(mean_test,  4),
                'STD Test':         round(std_test,   4),
                'Old Benchmark':    old_score,
                'Diff':             diff,
            }
            rows_s5.append(row)

            save_results(
                pd.DataFrame([row]),
                filename=f'{base}_{ts_now}.csv',
                extra_json=json_payload,
                json_filename=f'{base}_{ts_now}.json',
            )

        except Exception:
            import traceback
            traceback.print_exc()
            print(f"  ERROR on {name} ({n_samples}) — see traceback above.")


print("\n" + "=" * 60)
print("SESSION 5 — DEEP-SSFS  |  FINAL CONSOLIDATED RESULTS")
print("=" * 60)
df_s5 = pd.DataFrame(rows_s5)
if not df_s5.empty:
    print(df_s5.to_string(index=False))
    ts_final = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    save_results(
        df_s5,
        filename=f'DeepSSFS_Session5_ALL_{ts_final}.csv',
        extra_json=best_hyperparams_s5,
        json_filename=f'DeepSSFS_Session5_ALL_{ts_final}.json',
    )
else:
    print("No results produced — check errors above.")

print(f"[{_ts()}] All results saved to {RESULTS_DIR}.")